In [ ]:
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
import os
import re


AudioSegment.converter = "D:/Github/phone-cleaner/bin/ffmpeg.exe"
AudioSegment.ffprobe = "D:/Github/phone-cleaner/bin/ffprobe.exe"
input_folder = "./phoneme-Samples/Glossika/wav-no-music/"
output_folder = "./phoneme-Samples/Glossika/tight-snips/"

# Create output directory if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

volume_threshold = -25  # dB
lead_time = 200  # milliseconds
follow_time = 200  # milliseconds

def find_name(input_string):
    # Use a regular expression to find the content within the first pair of square brackets
    match = re.search(r'\] ([^\]]+)\.', input_string)
    
    # If a match is found, trim leading and trailing spaces
    if match:
        return match.group(1).strip()
    else:
        return None
    
def find_bracket_contents(input_string):
    # Use a regular expression to find the content within the first pair of square brackets
    match = re.search(r'\[([^\]]+)\]', input_string)
    
    # If a match is found, trim leading and trailing spaces
    if match:
        return match.group(1).strip()
    else:
        return None

In [9]:
#Clipping Glossika
for filename in os.listdir(input_folder):
    
    if filename.endswith(".wav"):
        audio_path = os.path.join(input_folder, filename)
        # Use Unicode string
        audio_path = audio_path
        
        
        print(audio_path)
         # Print the audio path to debug
        print("Processing file:", audio_path)
        
        # Convert the file to a standard format
        temp_audio_path = os.path.join(output_folder, "temp_output.wav")
        start_time = "00:00:25"  # 25 seconds
        end_time = "00:00:50"    # 50 seconds
        
        conversion_command = f'ffmpeg.exe -y -ss {start_time} -to {end_time} -i "{audio_path}" -acodec pcm_s16le -ar 44100 "{temp_audio_path}"'
        
        print(f"Running command: {conversion_command}")
        result = os.system(conversion_command)
        print(f"Command result: {result}")
        
        # Check if temp file was created
        if not os.path.exists(temp_audio_path):
            print(f"Error: temp file not created at {temp_audio_path}")
            continue
        
        try:
            # Load the converted audio file
            audio = AudioSegment.from_file(temp_audio_path, format="wav")
        except Exception as e:
            print("Error loading audio file:", e)
            continue
        
        # Detect nonsilent segments
        nonsilent_ranges = detect_nonsilent(audio, min_silence_len=300, silence_thresh=volume_threshold)
        phoneme = find_bracket_contents(filename)
        name = find_name(filename)
        for i, (start, end) in enumerate(nonsilent_ranges):
            start = max(0, start - lead_time)
            end = min(len(audio), end + follow_time)
            clip = audio[start:end]
            output_filename = f"{os.path.splitext(filename)[0]}_clip_{i}.wav"
            if i == 0:
                output_filename = f"iso_[{phoneme}]_{phoneme}_{name}.wav"
            if i == 1:
                output_filename = f"pre_[{phoneme}]_{phoneme}ə_{name}.wav"
            if i == 2:
                output_filename = f"med_[{phoneme}]_ə{phoneme}ə_{name}.wav"
            if i == 3:
                output_filename = f"post_[{phoneme}]_ə{phoneme}_{name}.wav"
                
            output_path = os.path.join(output_folder, output_filename)
            if ( 0 <= i <= 3):
                # Print the output path to debug
                print("Exporting clip to:", output_path)

                try:
                    # Export clip with Unicode handling
                    if ("_clean" in output_path):
                        output_path = output_path.replace("_clean", "")
                    clip.export(output_path, format="wav")
                except Exception as e:
                    print("Error exporting file:", e)
                    continue

                
# 0 Music 1
# 1 Music 2
# 2 Isolated 1
# 3 Isolated 2
# 4 Xə
# 5 əXə
# 6 əX
# 7 Examples
# 8 Examples
# 9 Examples

./phoneme-Samples/Glossika/wav-no-music/[ b ] voiced unaspirated bilabial stop_clean.wav
Processing file: ./phoneme-Samples/Glossika/wav-no-music/[ b ] voiced unaspirated bilabial stop_clean.wav
Running command: ffmpeg.exe -y -ss 00:00:25 -to 00:00:50 -i "./phoneme-Samples/Glossika/wav-no-music/[ b ] voiced unaspirated bilabial stop_clean.wav" -acodec pcm_s16le -ar 44100 "./phoneme-Samples/Glossika/tight-snips/temp_output.wav"
Command result: 0
Exporting clip to: ./phoneme-Samples/Glossika/tight-snips/iso_[b]_b_voiced unaspirated bilabial stop_clean.wav
Exporting clip to: ./phoneme-Samples/Glossika/tight-snips/pre_[b]_bə_voiced unaspirated bilabial stop_clean.wav
Exporting clip to: ./phoneme-Samples/Glossika/tight-snips/med_[b]_əbə_voiced unaspirated bilabial stop_clean.wav
Exporting clip to: ./phoneme-Samples/Glossika/tight-snips/post_[b]_əb_voiced unaspirated bilabial stop_clean.wav
./phoneme-Samples/Glossika/wav-no-music/[ bʱ ] voiced aspirated bilabial stop_clean.wav
Processing fil

In [4]:
#Clipping IPA

# Waiting to clean up inconsistent segments
verbose = False

input_folder = "C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV/"
output_folder = "C:/Github/phone-cleaner/phoneme-Samples/Glossika/tight-snips/"

for filename in os.listdir(input_folder):
    
    if filename.endswith(".wav"):
        audio_path = os.path.join(input_folder, filename)
        # Use Unicode string
        audio_path = audio_path
        
        
        print(audio_path)
         # Print the audio path to debug
        if verbose: 
            print("Processing file:", audio_path)
        
        # Convert the file to a standard format
        temp_audio_path = os.path.join(output_folder, "temp_output.wav")
        
        conversion_command = f'ffmpeg -y -i "{audio_path}" -acodec pcm_s16le -ar 44100 "{temp_audio_path}"'
        
        os.system(conversion_command)
        
        try:
            # Load the converted audio file
            audio = AudioSegment.from_file(temp_audio_path, format="wav")
        except Exception as e:
            print("Error loading audio file:", e)
            continue
        
        # Detect nonsilent segments
        nonsilent_ranges = detect_nonsilent(audio, min_silence_len=100, silence_thresh=volume_threshold)
        phoneme = find_bracket_contents(filename)
        name = find_name(filename)
        for i, (start, end) in enumerate(nonsilent_ranges):
            start = max(0, start - lead_time)
            end = min(len(audio), end + follow_time)
            clip = audio[start:end]
            output_filename = f"{os.path.splitext(filename)[0]}_clip_{i}.wav"
            if i == 0:
                output_filename = f"iso_{filename}_clip_{i}" ############Filename
                if verbose:
                    print(output_filename)
            if i == 1:
                output_filename = f"x_{filename}_clip_{i}"
                print(output_filename)
            if i == 2:
                output_filename = f"x_{filename}_clip_{i}"
                print(output_filename)
            output_path = os.path.join(output_folder, output_filename)
            if ( 0 <= i <= 3):
                # Print the output path to debug
                if verbose:
                    print("Exporting clip to:", output_path)
                try:
                    # Export clip with Unicode handling
                    if True == True:
                        clip.export(output_path, format="wav")
                except Exception as e:
                    print("Error exporting file:", e)
                    continue


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV/'

In [ ]:
import os
import subprocess
import json

def get_audio_details(file_path):
    command = [
        'ffprobe',
        '-v', 'error',
        '-show_format',
        '-show_streams',
        '-print_format', 'json',
        file_path
    ]
    
    result = subprocess.run(command, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"Error getting details for {file_path}: {result.stderr}")
        return None
    
    return json.loads(result.stdout)

def convert_webm_to_wav(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Get a list of all .webm files in the input folder
    webm_files = [f for f in os.listdir(input_folder) if f.endswith('.webm')]

    # Iterate over each .webm file and convert to WAV
    for webm_file in webm_files:
        input_path = os.path.join(input_folder, webm_file)
        output_file = os.path.splitext(webm_file)[0] + '.wav'
        output_path = os.path.join(output_folder, output_file)

        # Ensure paths are correctly formatted for subprocess
        input_path = os.path.abspath(input_path)
        output_path = os.path.abspath(output_path)

        # FFMPEG command for conversion
        command = [
            'ffmpeg',
            '-i', input_path,
            '-acodec', 'pcm_s16le',  # 16-bit little-endian PCM audio
            '-ar', '44100',
            '-f', 'wav',
            output_path
        ]

        # Print command for debugging
        print(f"Running command: {' '.join(command)}")

        # Execute the FFMPEG command
        result = subprocess.run(command, capture_output=True, text=True)

        # Print stdout and stderr for debugging
        print("stdout:", result.stdout)
        print("stderr:", result.stderr)

        # Check if the output file was created successfully
        if os.path.exists(output_path):
            print(f'Conversion complete: {webm_file} -> {output_file}')
            
            # Get and print audio details using ffprobe
            details = get_audio_details(output_path)
            if details:
                print(f"Audio details for {output_file}:")
                print(json.dumps(det


In [1]:
## Convert IPA to Wav

import os
from pydub import AudioSegment

def convert_mp3_to_wav(input_folder, output_folder):
    # Ensure output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Loop through all files in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith(".mp3"):
            mp3_path = os.path.join(input_folder, filename)
            wav_path = os.path.join(output_folder, os.path.splitext(filename)[0] + ".wav")
            
            # Load the MP3 file
            audio = AudioSegment.from_mp3(mp3_path)
            
            # Export as WAV with PCM 16-bit LE encoding
            audio.export(wav_path, format="wav", codec="pcm_s16le")

            print(f"Converted {filename} to {wav_path}")

paths = [
    "C:/Github/phone-cleaner/phoneme-Samples/IPA/JE", 
    "C:/Github/phone-cleaner/phoneme-Samples/IPA/JW",
    "C:/Github/phone-cleaner/phoneme-Samples/IPA/JH",
    "C:/Github/phone-cleaner/phoneme-Samples/IPA/PL"
]

output_folder = "C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV"
for input_folder in paths:
    convert_mp3_to_wav(input_folder, output_folder)






Converted iso_[a]_a_OPEN FRONT UNROUNDED VOWEL_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[a]_a_OPEN FRONT UNROUNDED VOWEL_IPAJE.wav
Converted iso_[a˞]_a˞_RHOTICITY_2_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[a˞]_a˞_RHOTICITY_2_IPAJE.wav
Converted iso_[a̤]_a̤_BREATHY VOICED_2_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[a̤]_a̤_BREATHY VOICED_2_IPAJE.wav
Converted iso_[a̰]_a̰_CREAKY VOICED_2_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[a̰]_a̰_CREAKY VOICED_2_IPAJE.wav
Converted iso_[b]_b_VOICED BILABIAL PLOSIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b]_b_VOICED BILABIAL PLOSIVE_IPAJE.wav
Converted iso_[b̤]_b̤_BREATHY VOICED_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b̤]_b̤_BREATHY VOICED_1_IPAJE.wav
Converted iso_[b̰]_b̰_CREAKY VOICED_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b̰]_b̰_CREAKY VOICED_1_IPAJ

Converted iso_[tʷ]_tʷ_LABIALIZED_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tʷ]_tʷ_LABIALIZED_1_IPAJE.wav
Converted iso_[tʼ]_tʼ_DENTAL or ALVEOLAR EJECTIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tʼ]_tʼ_DENTAL or ALVEOLAR EJECTIVE_IPAJE.wav
Converted iso_[tˤ]_tˤ_PHARYNGEALIZED_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tˤ]_tˤ_PHARYNGEALIZED_1_IPAJE.wav
Converted iso_[t̪]_t̪_DENTAL_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̪]_t̪_DENTAL_1_IPAJE.wav
Converted iso_[t̬]_t̬_VOICED_2_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̬]_t̬_VOICED_2_IPAJE.wav
Converted iso_[t̺]_t̺_APICAL_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̺]_t̺_APICAL_1_IPAJE.wav
Converted iso_[t̻]_t̻_LAMINAL_1_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̻]_t̻_LAMINAL_1_IPAJE.wav
Converted iso_[t̼]_t̼_LINGUOLABIAL_1_IPAJE.mp3 to C:/G

Converted iso_[ɢ]_ɢ_VOICED UVULAR PLOSIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɢ]_ɢ_VOICED UVULAR PLOSIVE_IPAJE.wav
Converted iso_[ɤ]_ɤ_CLOSE-MID BACK UNROUNDED VOWEL_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɤ]_ɤ_CLOSE-MID BACK UNROUNDED VOWEL_IPAJE.wav
Converted iso_[ɥ]_ɥ_VOICED LABIAL-PALATAL APPROXIMANT_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɥ]_ɥ_VOICED LABIAL-PALATAL APPROXIMANT_IPAJE.wav
Converted iso_[ɦ]_ɦ_VOICED GLOTTAL FRICATIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɦ]_ɦ_VOICED GLOTTAL FRICATIVE_IPAJE.wav
Converted iso_[ɧ]_ɧ_VOICELESS POSTALVEOLAR-VELAR FRICATIVE_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɧ]_ɧ_VOICELESS POSTALVEOLAR-VELAR FRICATIVE_IPAJE.wav
Converted iso_[ɪ]_ɪ_NEAR-CLOSE NEAR-FRONT UNROUNDED VOWEL_IPAJE.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɪ]_ɪ_NEAR-CLOSE NEAR-FRONT UNROUNDED VOWEL_IPAJE.wa

Converted iso_[b̤a̤]_b̤a̤_BREATHY VOICED_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b̤a̤]_b̤a̤_BREATHY VOICED_IPAJW.wav
Converted iso_[b̰a̰]_b̰a̰_CREAKY VOICED_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[b̰a̰]_b̰a̰_CREAKY VOICED_IPAJW.wav
Converted iso_[c]_c_VOICELESS PALATAL PLOSIVE_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[c]_c_VOICELESS PALATAL PLOSIVE_IPAJW.wav
Converted iso_[d]_d_VOICED DENTAL or ALVEOLAR PLOSIVE_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d]_d_VOICED DENTAL or ALVEOLAR PLOSIVE_IPAJW.wav
Converted iso_[dʰ]_dʰ_ASPIRATED_2_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[dʰ]_dʰ_ASPIRATED_2_IPAJW.wav
Converted iso_[dʲ]_dʲ_PALATALIZED_2_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[dʲ]_dʲ_PALATALIZED_2_IPAJW.wav
Converted iso_[dʷ]_dʷ_LABIALIZED_2_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[dʷ]_dʷ_LAB

Converted iso_[tˠ]_tˠ_VELARIZED_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tˠ]_tˠ_VELARIZED_1_IPAJW.wav
Converted iso_[tˤ]_tˤ_PHARYNGEALIZED_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[tˤ]_tˤ_PHARYNGEALIZED_1_IPAJW.wav
Converted iso_[t̪]_t̪_DENTAL_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̪]_t̪_DENTAL_1_IPAJW.wav
Converted iso_[t̺]_t̺_APICAL_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̺]_t̺_APICAL_1_IPAJW.wav
Converted iso_[t̻]_t̻_LAMINAL_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̻]_t̻_LAMINAL_1_IPAJW.wav
Converted iso_[t̼]_t̼_LINGUOLABIAL_1_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t̼]_t̼_LINGUOLABIAL_1_IPAJW.wav
Converted iso_[t͜s]_t͜s_TIE BAR (BELOW)_IPAJW.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t͜s]_t͜s_TIE BAR (BELOW)_IPAJW.wav
Converted iso_[u]_u_CLOSE BACK ROUNDED VOWEL_IPAJW.mp3 to C:/Git

...

Converted iso_[x]_x_VOICELESS VELAR FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[x]_x_VOICELESS VELAR FRICATIVE_IPAJH.wav
Converted iso_[y]_y_CLOSE FRONT ROUNDED VOWEL_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[y]_y_CLOSE FRONT ROUNDED VOWEL_IPAJH.wav
Converted iso_[z]_z_VOICED ALVEOLAR FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[z]_z_VOICED ALVEOLAR FRICATIVE_IPAJH.wav
Converted iso_[æ]_æ_NEAR-OPEN FRONT UNROUNDED VOWEL_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[æ]_æ_NEAR-OPEN FRONT UNROUNDED VOWEL_IPAJH.wav
Converted iso_[ç]_ç_VOICELESS PALATAL FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ç]_ç_VOICELESS PALATAL FRICATIVE_IPAJH.wav
Converted iso_[ð]_ð_VOICED DENTAL FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ð]_ð_VOICED DENTAL FRICATIVE_IPAJH.wav
Converted iso_[ø]_ø_CLOSE-MID FRONT ROUNDED VOWEL_IP

Converted iso_[ɴ]_ɴ_VOICED UVULAR NASAL_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɴ]_ɴ_VOICED UVULAR NASAL_IPAJH.wav
Converted iso_[ɶ]_ɶ_OPEN FRONT ROUNDED VOWEL_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɶ]_ɶ_OPEN FRONT ROUNDED VOWEL_IPAJH.wav
Converted iso_[ɸ]_ɸ_VOICELESS BILABIAL FRICATIVE_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɸ]_ɸ_VOICELESS BILABIAL FRICATIVE_IPAJH.wav
Converted iso_[ɹ]_ɹ_VOICED DENTAL or ALVEOLAR APPROXIMANT_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɹ]_ɹ_VOICED DENTAL or ALVEOLAR APPROXIMANT_IPAJH.wav
Converted iso_[ɹ̝]_ɹ̝_RAISED_2_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɹ̝]_ɹ̝_RAISED_2_IPAJH.wav
Converted iso_[ɺ]_ɺ_VOICED ALVEOLAR LATERAL FLAP_IPAJH.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɺ]_ɺ_VOICED ALVEOLAR LATERAL FLAP_IPAJH.wav
Converted iso_[ɻ]_ɻ_VOICED RETROFLEX APPROXIMANT_IPAJH.mp3 to C:/Github/

Converted iso_[d̥]_d̥_VOICELESS_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̥]_d̥_VOICELESS_2_IPAPL.wav
Converted iso_[d̪]_d̪_DENTAL_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̪]_d̪_DENTAL_2_IPAPL.wav
Converted iso_[d̺]_d̺_APICAL_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̺]_d̺_APICAL_2_IPAPL.wav
Converted iso_[d̻]_d̻_LAMINAL_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̻]_d̻_LAMINAL_2_IPAPL.wav
Converted iso_[d̼]_d̼_LINGUOLABIAL_2_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[d̼]_d̼_LINGUOLABIAL_2_IPAPL.wav
Converted iso_[e]_e_CLOSE-MID FRONT UNROUNDED VOWEL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[e]_e_CLOSE-MID FRONT UNROUNDED VOWEL_IPAPL.wav
Converted iso_[eː]_eː_LONG_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[eː]_eː_LONG_IPAPL.wav
Converted iso_[eˑ]_eˑ_HALF-LONG_IPAPL.mp3 to C:/Github/phone-cle

Converted iso_[t͜s]_t͜s_TIE BAR (BELOW)_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[t͜s]_t͜s_TIE BAR (BELOW)_IPAPL.wav
Converted iso_[u]_u_CLOSE BACK ROUNDED VOWEL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[u]_u_CLOSE BACK ROUNDED VOWEL_IPAPL.wav
Converted iso_[u̟]_u̟_ADVANCED_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[u̟]_u̟_ADVANCED_IPAPL.wav
Converted iso_[v]_v_VOICED LABIODENTAL FRICATIVE_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[v]_v_VOICED LABIODENTAL FRICATIVE_IPAPL.wav
Converted iso_[w]_w_VOICED LABIAL-VELAR APPROXIMANT_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[w]_w_VOICED LABIAL-VELAR APPROXIMANT_IPAPL.wav
Converted iso_[x]_x_VOICELESS VELAR FRICATIVE_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[x]_x_VOICELESS VELAR FRICATIVE_IPAPL.wav
Converted iso_[y]_y_CLOSE FRONT ROUNDED VOWEL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme

Converted iso_[ɪ]_ɪ_NEAR-CLOSE NEAR-FRONT UNROUNDED VOWEL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɪ]_ɪ_NEAR-CLOSE NEAR-FRONT UNROUNDED VOWEL_IPAPL.wav
Converted iso_[ɬ]_ɬ_VOICELESS DENTAL or ALVEOLAR LATERAL FRICATIVE_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɬ]_ɬ_VOICELESS DENTAL or ALVEOLAR LATERAL FRICATIVE_IPAPL.wav
Converted iso_[ɭ]_ɭ_VOICED RETROFLEX LATERAL APPROXIMANT_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɭ]_ɭ_VOICED RETROFLEX LATERAL APPROXIMANT_IPAPL.wav
Converted iso_[ɮ]_ɮ_VOICED DENTAL or ALVEOLAR LATERAL FRICATIVE_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɮ]_ɮ_VOICED DENTAL or ALVEOLAR LATERAL FRICATIVE_IPAPL.wav
Converted iso_[ɰ]_ɰ_VOICED VELAR APPROXIMANT_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IPA/IPAWAV\iso_[ɰ]_ɰ_VOICED VELAR APPROXIMANT_IPAPL.wav
Converted iso_[ɳ]_ɳ_VOICED RETROFLEX NASAL_IPAPL.mp3 to C:/Github/phone-cleaner/phoneme-Samples/IP